In [2]:
import pandas as pd

In [21]:
df_pedro = pd.read_csv('../data/processed/datos_pedro.csv', sep='|', index_col=0)
df_mili = pd.read_csv('../data/processed/pib_vs_calidad_vida.csv', sep=',', index_col=0).reset_index()
df_manu = pd.read_csv('../data/interim/alquiler_venta_ccaa.csv', sep=',', index_col=0)

# Renombramos columnas de comunidades autónomas y año para realizar el merge en ellas
df_mili = df_mili.rename(columns = {'Comunidad Autónoma':'com_aut'})
df_manu = df_manu.rename(columns = {'Año':'año', 'Comunidad Autonoma':'com_aut'})


In [22]:
MAP = {'Andalucia':'Andalucía',
       'Aragon':'Aragón',
       'Baleares':'Balears' ,
       'Castilla-La Mancha':'Castilla - La Mancha',
       'Comunidad Valenciana':'Comunitat Valenciana',
       'La Rioja':'Rioja',
       'Euskadi': 'País Vasco'}
for i,j in MAP.items():
    df_manu["com_aut"] = df_manu["com_aut"].replace(i,j)

In [33]:
def merge_datasets(df_left:pd.DataFrame, df_right:pd.DataFrame, on1_left:str, on1_right:str, on2_left:str, on2_right:str):
    '''
    Función para hacer merge de dos datasets con las mismas referencias ordenadas pero con variaciones en el nombre. Fuerza el cambio de nombres para hacer merge.
    '''
    values_left = df_left[on1_left].sort_values().unique()
    values_right = df_right[on1_right].sort_values().unique()
    dicc = {values_right[i]: values_left[i] for i in range(len(values_right))}
    df_right[on1_right] = df_right[on1_right].map(dicc)

    df_right = df_right.rename(columns={on1_right: on1_left})
    df_right = df_right.rename(columns={on2_right: on2_left})

    return pd.merge(df_left, df_right, 'outer', [on1_left, on2_left])

In [35]:
df_pedro_mili = merge_datasets(df_pedro, df_mili, 'com_aut', 'com_aut', 'año', 'año')

In [37]:
df_final = merge_datasets(df_pedro_mili , df_manu, 'com_aut', 'com_aut', 'año', 'año')

In [39]:
df_final.head()

,año,com_aut,pib,pob,sui,nat,paro,ing,pobr,fum,...,retrasos_pagos(%),renta_media,renta_mediana,riesgo_pobreza(%),dificultad_fin_mes(%),desigualdad_ing(S80/S20),inc_gastos_imprevistos(%),tasa_criminalidad,precio_medio_anual_eur_m2_venta,precio_medio_anual_eur_m2_alquiler
0,2000,Andalucía,86760722.0,7301.8,NaN,11.08,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2001,Andalucía,93964317.0,7341.5,11.17,11.06,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2002,Andalucía,101269401.0,7437.2,10.36,11.01,NaN,NaN,NaN,31.84,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2003,Andalucía,110093641.0,7544.2,10.35,11.42,NaN,NaN,NaN,30.39,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2004,Andalucía,119026615.0,7649.0,11.01,11.64,NaN,NaN,NaN,28.82,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
df_final.com_aut.nunique()

19

In [50]:
len(df_final.columns[2:]) #29 columnas sin contar año y comunidades

29

In [58]:
for i in [df_pedro,df_mili,df_manu]:
    cnt_columnas = len(i.columns[2:])
    print(cnt_columnas)

14
13
2


In [59]:
df_final.isna().sum()

año                                     0
com_aut                                 0
pib                                    19
pob                                    19
sui                                    38
nat                                    38
paro                                  152
ing                                   190
pobr                                  304
fum                                   323
alc                                   361
obes                                  323
fyv                                   361
crim                                  224
homi                                  224
sati                                  380
pib_pc                                209
renta_pc                              209
poblacion                             209
gasto_elevado_vivienda(%)             209
falta_espacio_vivienda(%)             209
retrasos_pagos(%)                     209
renta_media                           209
renta_mediana                     

In [60]:
df = df_final[(df_final["año"] > 2007) & (df_final["año"] < 2023)].copy()

In [63]:
df.columns

Index(['año', 'com_aut', 'pib', 'pob', 'sui', 'nat', 'paro', 'ing', 'pobr',
       'fum', 'alc', 'obes', 'fyv', 'crim', 'homi', 'sati', 'pib_pc',
       'renta_pc', 'poblacion', 'gasto_elevado_vivienda(%)',
       'falta_espacio_vivienda(%)', 'retrasos_pagos(%)', 'renta_media',
       'renta_mediana', 'riesgo_pobreza(%)', 'dificultad_fin_mes(%)',
       'desigualdad_ing(S80/S20)', 'inc_gastos_imprevistos(%)',
       'tasa_criminalidad', 'precio_medio_anual_eur_m2_venta',
       'precio_medio_anual_eur_m2_alquiler'],
      dtype='object')

**La importancia incial se mide clasificándolas en target/directora (0), agrupación importante (1), agrupación interesante (2), agrupación secundaria (3)**  
0 (directoras): estructuran el análisis temporal y territorial.  
1 (núcleo): variables económicas y de bienestar directamente relacionadas con el PIB.  
2 (impacto social): reflejan consecuencias o mediadores del nivel económico.  
3 (contexto): variables de control y descriptivas.  

| Variable | Descripción | Tipo de variable | Importancia inicial |
|--------|------------|-----------------|---------------------|
| año | Año de referencia de los datos. | Numérica discreta | 0 |
| com_aut | Comunidad Autónoma de España. | Categórica nominal | 0 |
| pib | Producto Interior Bruto. Valor económico agregado. | Numérica continua | 1 |
| pob | Población total de la comunidad. | Numérica discreta | 3 |
| sui | Tasa de suicidios. Defunciones por suicidio por 100.000 habitantes. | Numérica continua | 2 |
| nat | Tasa de natalidad. Nacimientos por cada 1.000 habitantes. | Numérica continua | 2 |
| paro | Tasa de desempleo. Parados sobre población activa. | Numérica continua | 1 |
| ing | Ingresos medios por persona adulta. | Numérica continua | 1 |
| pobr | Tasa de riesgo de pobreza. | Numérica continua | 1 |
| fum | Porcentaje de población adulta fumadora. | Numérica continua | 3 |
| alc | Porcentaje de población adulta con consumo de riesgo de alcohol. | Numérica continua | 3 |
| obes | Porcentaje de población adulta con obesidad. | Numérica continua | 3 |
| fyv | Porcentaje de población con consumo diario de frutas y verduras. | Numérica continua | 3 |
| crim | Tasa de criminalidad. Infracciones penales por 1.000 habitantes. | Numérica continua | 2 |
| homi | Tasa de homicidios. Homicidios y asesinatos por 100.000 habitantes. | Numérica continua | 3 |
| sati | Nivel de satisfacción general con la vida. | Numérica continua | 1 |
| pib_pc | PIB per cápita a precios de mercado. | Numérica continua | 1 |
| renta_pc | Renta disponible bruta de los hogares per cápita. | Numérica continua | 1 |
| poblacion | Número total de habitantes por comunidad y año. | Numérica discreta | 3 |
| gasto_elevado_vivienda(%) | Porcentaje de población con gasto elevado en vivienda. | Numérica continua | 2 |
| falta_espacio_vivienda(%) | Porcentaje de población que vive en viviendas con falta de espacio. | Numérica continua | 3 |
| retrasos_pagos(%) | Porcentaje de población con retrasos en pagos de vivienda o suministros. | Numérica continua | 2 |
| renta_media | Renta media por unidad de consumo. | Numérica continua | 1 |
| renta_mediana | Renta mediana por unidad de consumo. | Numérica continua | 1 |
| riesgo_pobreza(%) | Porcentaje de población en riesgo de pobreza. | Numérica continua | 1 |
| dificultad_fin_mes(%) | Porcentaje de población con dificultad para llegar a fin de mes. | Numérica continua | 2 |
| desigualdad_ing(S80/S20) | Cociente de desigualdad entre el 20% más rico y el 20% más pobre. | Numérica continua | 1 |
| inc_gastos_imprevistos(%) | Porcentaje de población que no puede afrontar gastos imprevistos. | Numérica continua | 2 |
| tasa_criminalidad | Delitos registrados por cada 1.000 habitantes. | Numérica continua | 2 |
| precio_medio_anual_eur_m2_venta | Precio medio anual de vivienda en venta por m² (€). | Numérica continua | 2 |
| precio_medio_anual_eur_m2_alquiler | Precio medio anual de vivienda en alquiler por m² (€). | Numérica continua | 2 |
